In [1]:
import pandas as pd

# Load metabolite data
mtb = pd.read_csv('fran_met.csv', index_col=0)

# Load metadata
metadata = pd.read_csv('fran_metadata .csv', index_col=0)

print("Metabolite data shape:", mtb.shape)
print("Metadata shape:", metadata.shape)

Metabolite data shape: (218, 1343)
Metadata shape: (218, 9)


In [2]:
# Look at first 5 rows of metadata
print(metadata.head())
print("\n")
# Look at disease group counts
print(metadata['Study.Group'].value_counts())

       Sample Study.Group  Age  Fecal.Calprotectin antibiotic  \
1  PRISM.7122          CD   38               207.0         No   
2  PRISM.7147          CD   50                 NaN         No   
3  PRISM.7150          CD   41               218.0         No   
4  PRISM.7153          CD   51                 NaN         No   
5  PRISM.7184          CD   68                20.0         No   

  immunosuppressant mesalamine steroids Disease.Group  
1               Yes         No       No           IBD  
2                No        Yes       No           IBD  
3               Yes         No       No           IBD  
4                No        Yes       No           IBD  
5                No         No       No           IBD  


Study.Group
CD         87
UC         75
Control    56
Name: count, dtype: int64


In [3]:
import pandas as pd

# Load metabolite data - index_col=0 means first column becomes row names
# This is exactly what Animesh did in R with rownames(mtb) <- mtb[,1]
mtb = pd.read_csv('fran_met.csv', index_col=0)

# Load metadata
metadata = pd.read_csv('fran_metadata .csv', index_col=0)

print("Metabolite data shape:", mtb.shape)
print("Metadata shape:", metadata.shape)
print("\nFirst 3 row names:", mtb.index[:3].tolist())

Metabolite data shape: (218, 1343)
Metadata shape: (218, 9)

First 3 row names: ['PRISM.7122', 'PRISM.7147', 'PRISM.7150']


In [4]:
# =============================================================
# STEP 2: SPARSITY FILTERING
# Converting Animesh's R code chunk to Python
# Original R code:
# sparsity_mtb <- c(rep(0,8849))
# for (i in 1:ncol(mtb)) {
#     sparsity_mtb[i] <- (sum(mtb[, i] == 0)/220)*100
# }
# remove_mtb <- which(sparsity_mtb > 80)
# mtb_alt <- mtb[-c(remove_mtb)]
# =============================================================

# For each metabolite (column), calculate what percentage
# of patients have a zero value (meaning metabolite not detected)
sparsity = (mtb == 0).sum(axis=0) / len(mtb) * 100

# Identify metabolites where more than 80% of patients have zero
# These metabolites are not useful for classification
remove = sparsity[sparsity > 80].index

# Remove the sparse metabolites from the dataset
mtb_alt = mtb.drop(columns=remove)

# Print results
print("Original number of metabolites:", mtb.shape[1])
print("Metabolites removed (>80% zeros):", len(remove))
print("Metabolites remaining:", mtb_alt.shape[1])

# NOTE: 0 metabolites removed because fran_met.csv is already
# Animesh's preprocessed output - sparsity filtering was already
# applied to the raw 8,849 metabolite dataset before saving


Original number of metabolites: 1343
Metabolites removed (>80% zeros): 0
Metabolites remaining: 1343


In [5]:
# =============================================================
# STEP 3: REMOVE UNKNOWN METABOLITES
# Converting Animesh's R code:
# remove <- which((is.na(mtb.map$HMDB)) & (is.na(mtb.map$KEGG))
#           & is.na(mtb.map$Compound.Name)
#           & is.na(mtb.map$Putative.Chemical.Class))
# mtb_alt <- mtb_alt[,c(-remove)]
# =============================================================

# EXPLANATION:
# Animesh removed metabolites that had NO identification at all:
# - No HMDB ID (Human Metabolome Database identifier)
# - No KEGG ID (Kyoto Encyclopedia of Genes and Genomes identifier)
# - No Compound Name
# - No Chemical Class
# These are completely unidentified signals - not useful for analysis

# NOTE: This step cannot be fully reproduced because the metabolite
# mapping file (mtb.map) containing HMDB/KEGG annotations is not
# publicly available on Animesh's GitHub.
# fran_met.csv is already the output AFTER this step was applied.
# Original: 8,835 metabolites → After removing unknowns: 3,935 metabolites

# We can verify our current data has no completely unnamed metabolites
# by checking column names - all should be identified compounds
unnamed = [col for col in mtb_alt.columns if col.startswith('C18') or col.startswith('C8')]

print("Number of potentially unidentified metabolites remaining:", len(unnamed))
print("Total metabolites in our dataset:", mtb_alt.shape[1])
print("\nNOTE: Animesh reduced from 8,835 to 3,935 metabolites in this step")
print("We are working from his already cleaned output (fran_met.csv)")

Number of potentially unidentified metabolites remaining: 674
Total metabolites in our dataset: 1343

NOTE: Animesh reduced from 8,835 to 3,935 metabolites in this step
We are working from his already cleaned output (fran_met.csv)


In [6]:
# =============================================================
# STEP 4: CLR TRANSFORMATION
# Converting Animesh's R code:
# tran_mtb <- clr(mtb_alt + 1)
# =============================================================

# We need scipy for this
import numpy as np

# EXPLANATION:
# CLR = Centred Log Ratio transformation
# Metabolomics data is compositional - values are relative to each other
# CLR fixes this by normalising each value against the geometric mean
# of all metabolites for that patient
# Adding 1 before CLR avoids log(0) error for zero values

def clr_transform(data):
    # Add 1 to avoid log(0) - same as Animesh's (mtb_alt + 1)
    data_plus1 = data + 1
    # Calculate geometric mean for each patient (each row)
    # Geometric mean = exp(mean of log values)
    geometric_mean = np.exp(np.log(data_plus1).mean(axis=1))
    # Divide each value by geometric mean and take log
    # This centres the data around zero
    clr_data = np.log(data_plus1.div(geometric_mean, axis=0))
    return clr_data

# Apply CLR transformation to our metabolite data
tran_mtb = clr_transform(mtb_alt)

print("Shape before CLR:", mtb_alt.shape)
print("Shape after CLR:", tran_mtb.shape)
print("\nBefore CLR - first value:", round(mtb_alt.iloc[0,0], 4))
print("After CLR - first value:", round(tran_mtb.iloc[0,0], 4))
print("\nAfter CLR - mean of all values (should be close to 0):", round(tran_mtb.values.mean(), 4))

Shape before CLR: (218, 1343)
Shape after CLR: (218, 1343)

Before CLR - first value: 88.0773
After CLR - first value: 1.0838

After CLR - mean of all values (should be close to 0): -0.0


In [7]:
# =============================================================
# STEP 5: Z-NORMALISATION (Standard Scaling)
# Converting Animesh's R code:
# mtb_norm <- as.data.frame(scale(tran_mtb))
# =============================================================

# Import StandardScaler from sklearn
# This does exactly what R's scale() function does
from sklearn.preprocessing import StandardScaler

# EXPLANATION:
# Z-normalisation transforms each metabolite so that:
# Mean = 0 and Standard Deviation = 1
# Formula: Z = (value - mean) / standard deviation
# This puts all metabolites on the same scale
# So ML models dont favour metabolites with larger numbers

# Create the scaler object
scaler = StandardScaler()

# Fit and transform the CLR transformed data
# fit = calculate mean and std for each metabolite
# transform = apply the formula to every value
mtb_norm = pd.DataFrame(
    scaler.fit_transform(tran_mtb),
    index=tran_mtb.index,
    columns=tran_mtb.columns
)

# Verify normalisation worked correctly
print("Shape after normalisation:", mtb_norm.shape)
print("\nBefore normalisation - mean:", round(tran_mtb.values.mean(), 4))
print("After normalisation - mean (should be ~0):", round(mtb_norm.values.mean(), 4))
print("\nBefore normalisation - std:", round(tran_mtb.values.std(), 4))
print("After normalisation - std (should be ~1):", round(mtb_norm.values.std(), 4))

Shape after normalisation: (218, 1343)

Before normalisation - mean: -0.0
After normalisation - mean (should be ~0): 0.0

Before normalisation - std: 2.4605
After normalisation - std (should be ~1): 1.0


In [8]:
# =============================================================
# STEP 6: REMOVE OUTLIER PATIENTS
# Converting Animesh's R code:
# met_diff <- mtb_alt[-c(126,182),]
# metadata_diff <- metadata[-c(126,182),]
# =============================================================

# EXPLANATION:
# Animesh identified 2 outlier patients from his PCoA plot
# on the microbiome (genera) data - rows 126 and 182
# These patients had metabolite profiles very different
# from all other patients and could distort the ML model
# He removed them from both metabolite and metadata datasets

# NOTE: In Python, row indexing starts at 0, not 1 like in R
# So R rows 126 and 182 = Python rows 125 and 181

# Check who these patients are before removing
print("Patient at row 126 (R) / 125 (Python):", mtb_norm.index[125])
print("Patient at row 182 (R) / 181 (Python):", mtb_norm.index[181])

# Remove outlier patients from metabolite data
met_diff = mtb_norm.drop(mtb_norm.index[[125, 181]])

# Remove same outlier patients from metadata
metadata_diff = metadata.drop(metadata.index[[125, 181]])

# Verify removal
print("\nMetabolite data before removal:", mtb_norm.shape)
print("Metabolite data after removal:", met_diff.shape)
print("Metadata before removal:", metadata.shape)
print("Metadata after removal:", metadata_diff.shape)

Patient at row 126 (R) / 125 (Python): PRISM.8803
Patient at row 182 (R) / 181 (Python): Validation.UMCG2136686

Metabolite data before removal: (218, 1343)
Metabolite data after removal: (216, 1343)
Metadata before removal: (218, 9)
Metadata after removal: (216, 9)


In [9]:
# =============================================================
# STEP 7: MANN-WHITNEY U TEST WITH HOCHBERG CORRECTION
# =============================================================

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# Fix: Reset index on both dataframes so they align properly
met_diff_reset = met_diff.reset_index(drop=True)
metadata_reset = metadata_diff.reset_index(drop=True)

# Get indices for IBD and Control patients
ibd_index = metadata_reset[metadata_reset['Disease.Group'] == 'IBD'].index
control_index = metadata_reset[metadata_reset['Disease.Group'] == 'Control'].index

print("IBD patients:", len(ibd_index))
print("Control patients:", len(control_index))

# Run Mann-Whitney U test for each metabolite
p_values = []
for metabolite in met_diff_reset.columns:
    ibd_values = met_diff_reset.loc[ibd_index, metabolite]
    control_values = met_diff_reset.loc[control_index, metabolite]
    stat, p = mannwhitneyu(ibd_values, control_values, alternative='two-sided')
    p_values.append(p)

print("\nTotal metabolites tested:", len(p_values))

# Apply Hochberg correction
reject, p_adjusted, _, _ = multipletests(p_values, method='fdr_bh', alpha=0.05)

# Keep only significant metabolites
significant = [met_diff_reset.columns[i] for i in range(len(p_adjusted)) if p_adjusted[i] < 0.05]
met_diff_sig = met_diff_reset[significant]

print("Metabolites after correction:", met_diff_sig.shape[1])
print("Metabolites removed:", met_diff_reset.shape[1] - met_diff_sig.shape[1])

IBD patients: 160
Control patients: 56

Total metabolites tested: 1343
Metabolites after correction: 897
Metabolites removed: 446


In [10]:
# =============================================================
# STEP 8: SAVE THE CLEANED DATA
# Converting Animesh's R code:
# write.csv(met_diff, "fran_met.csv", row.names = TRUE)
# =============================================================

# EXPLANATION:
# Save the final preprocessed metabolite data as a CSV file
# This file will be used for ML models and visualisations
# row.names = TRUE in R means we keep the patient sample names
# In Python, index=True does the same thing

# Save cleaned metabolite data
met_diff_sig.to_csv('fran_met_cleaned.csv', index=True)

# Save cleaned metadata
metadata_diff.to_csv('fran_metadata_cleaned.csv', index=True)

print("Saved cleaned metabolite data as: fran_met_cleaned.csv")
print("Saved cleaned metadata as: fran_metadata_cleaned.csv")
print("\nFinal metabolite data shape:", met_diff_sig.shape)
print("Final metadata shape:", metadata_diff.shape)
print("\nSummary of preprocessing pipeline:")
print("Original metabolites: 1,343")
print("After Mann-Whitney U test: 897")
print("Final patients: 216")
print("Final classes: CD, UC, Control")

Saved cleaned metabolite data as: fran_met_cleaned.csv
Saved cleaned metadata as: fran_metadata_cleaned.csv

Final metabolite data shape: (216, 897)
Final metadata shape: (216, 9)

Summary of preprocessing pipeline:
Original metabolites: 1,343
After Mann-Whitney U test: 897
Final patients: 216
Final classes: CD, UC, Control


In [11]:
# =============================================================
# STEP 9: BASELINE CHARACTERISTICS TABLE
# Converting Animesh's R code:
# CreateCatTable(vars = c("mesalamine", "antibiotic",
#               "immunosuppressant", "steroids"),
#               strata = c("Disease.Group"))
# t_Test <- t.test(Age ~ Disease.Group, data = metadata_diff)
# =============================================================

from scipy.stats import chi2_contingency, ttest_ind
import numpy as np

# EXPLANATION:
# Table 1 summarises patient demographics and medications
# Split by Disease Group (IBD vs Control)
# For medications (categorical) - use Chi-square test
# For age (continuous) - use t-test
# p-value < 0.05 means significant difference between groups

# --- MEDICATION SUMMARY ---
medications = ['antibiotic', 'immunosuppressant', 'mesalamine', 'steroids']

print("=" * 65)
print(f"{'Variable':<20} {'IBD (n=160)':<20} {'Control (n=56)':<15} {'p-value'}")
print("=" * 65)

for med in medications:
    # Count Yes/No for each group
    ibd_yes = (metadata_diff[metadata_diff['Disease.Group'] == 'IBD'][med] == 'Yes').sum()
    ibd_no = (metadata_diff[metadata_diff['Disease.Group'] == 'IBD'][med] == 'No').sum()
    con_yes = (metadata_diff[metadata_diff['Disease.Group'] == 'Control'][med] == 'Yes').sum()
    con_no = (metadata_diff[metadata_diff['Disease.Group'] == 'Control'][med] == 'No').sum()

    # Calculate percentages
    ibd_pct = round(ibd_yes / (ibd_yes + ibd_no) * 100, 1)
    con_pct = round(con_yes / (con_yes + con_no) * 100, 1)

    # Chi-square test for significance
    contingency = [[ibd_yes, ibd_no], [con_yes, con_no]]
    chi2, p, _, _ = chi2_contingency(contingency)

    print(f"{med:<20} {ibd_yes} ({ibd_pct}%){'':<10} {con_yes} ({con_pct}%){'':<5} {round(p, 4)}")

print("=" * 65)

# --- AGE SUMMARY ---
ibd_age = metadata_diff[metadata_diff['Disease.Group'] == 'IBD']['Age'].dropna()
con_age = metadata_diff[metadata_diff['Disease.Group'] == 'Control']['Age'].dropna()

# t-test for age difference
t_stat, p_age = ttest_ind(ibd_age, con_age)

print(f"\nAge (mean ± std):")
print(f"IBD:     {round(ibd_age.mean(),1)} ± {round(ibd_age.std(),1)}")
print(f"Control: {round(con_age.mean(),1)} ± {round(con_age.std(),1)}")
print(f"p-value: {round(p_age, 4)}")
print("=" * 65)

Variable             IBD (n=160)          Control (n=56)  p-value
antibiotic           18 (11.5%)           0 (0.0%)      0.0179
immunosuppressant    66 (41.2%)           0 (0.0%)      0.0
mesalamine           61 (38.4%)           0 (0.0%)      0.0
steroids             38 (24.1%)           0 (0.0%)      0.0031

Age (mean ± std):
IBD:     44.2 ± 17.0
Control: 37.0 ± 13.3
p-value: 0.0043
